# 🚀 TraderPath TFT Model Training

Bu notebook ile TFT modelini Google Colab'da eğitebilirsiniz.

**Adımlar:**
1. GPU'yu aktif et (Runtime → Change runtime type → T4 GPU)
2. Tüm hücreleri sırayla çalıştır
3. Eğitilen modeli indir

## 1. GPU Kontrolü

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU bulunamadı! Runtime → Change runtime type → T4 GPU seçin.")

## 2. Bağımlılıkları Yükle

In [ ]:
!pip install -q pytorch-forecasting pytorch-lightning optuna pandas numpy httpx loguru

## 3. GitHub'dan Kodu Çek

In [ ]:
!git clone https://github.com/yuksel-arslan/traderpath.git
%cd traderpath/services/tft-predictor

## 4. Eğitim Ayarları

In [ ]:
# ===== AYARLAR =====
TRADE_TYPE = "swing"  # scalp, swing, position
SYMBOLS = ["BTC", "ETH", "SOL", "BNB", "XRP"]
EPOCHS = 50
BATCH_SIZE = 64
SKIP_OPTUNA = True  # Hızlı test için True, tam eğitim için False

print(f"Trade Type: {TRADE_TYPE}")
print(f"Symbols: {SYMBOLS}")
print(f"Epochs: {EPOCHS}")

## 5. Eğitimi Başlat

In [ ]:
import sys
sys.path.insert(0, 'src')

from training import TradeType, TrainingConfig, TFTTrainer

# Trade type mapping
trade_type_map = {
    'scalp': TradeType.SCALP,
    'swing': TradeType.SWING,
    'position': TradeType.POSITION
}

# Config oluştur
config = TrainingConfig(
    symbols=SYMBOLS,
    trade_type=trade_type_map[TRADE_TYPE],
    max_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    use_gpu=True,
    n_optuna_trials=20 if not SKIP_OPTUNA else 0
)

print(f"\nConfig:")
print(f"  Data interval: {config.trade_type_config.data_interval}")
print(f"  Lookback days: {config.trade_type_config.lookback_days}")
print(f"  Prediction horizons: {config.trade_type_config.prediction_horizons}")

In [ ]:
# Trainer oluştur
trainer = TFTTrainer(config)

# Veri çek
print("📥 Binance'den veri çekiliyor...")
await trainer.fetch_data()
print(f"✅ {len(trainer.raw_data)} sembol için veri çekildi")

In [ ]:
# Feature engineering
print("🔧 Feature engineering...")
trainer.engineer_features()
print(f"✅ {len(trainer.processed_data)} sample, {len(trainer.metadata['time_varying_unknown'])} feature")

In [ ]:
# Hyperparameter optimization (opsiyonel)
if not SKIP_OPTUNA:
    print("🔍 Optuna ile hyperparameter optimization...")
    trainer.optimize_hyperparameters(n_trials=20)
else:
    print("⏭️ Optuna atlandı, varsayılan parametreler kullanılacak")

In [ ]:
# Model eğitimi
print("🚀 Model eğitimi başlıyor...")
trainer.train_final_model()
print("✅ Eğitim tamamlandı!")

In [ ]:
# Backtest
print("📊 Backtest çalıştırılıyor...")
backtest_result = trainer.run_backtest()

print(f"\n📈 BACKTEST SONUÇLARI:")
print(f"  Total Trades: {backtest_result.total_trades}")
print(f"  Win Rate: {backtest_result.win_rate:.2%}")
print(f"  Sharpe Ratio: {backtest_result.sharpe_ratio:.2f}")
print(f"  Max Drawdown: {backtest_result.max_drawdown:.2%}")
print(f"  Direction Accuracy: {backtest_result.direction_accuracy:.2%}")

## 6. Modeli Kaydet ve İndir

In [ ]:
# Modeli kaydet
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
model_filename = f"tft_{TRADE_TYPE}_{timestamp}.pt"

model_path = trainer.save_model(model_filename)
print(f"✅ Model kaydedildi: {model_path}")

In [ ]:
# Google Drive'a kaydet (opsiyonel)
from google.colab import drive
drive.mount('/content/drive')

import shutil
drive_path = f"/content/drive/MyDrive/traderpath_models/{model_filename}"
shutil.copy(model_path, drive_path)
print(f"✅ Model Drive'a kaydedildi: {drive_path}")

In [ ]:
# Doğrudan indir
from google.colab import files
files.download(model_path)
print("📥 Model indiriliyor...")

## 7. Özet

Eğitim tamamlandı! Şimdi:

1. **Model dosyasını** Railway'deki TFT servisine yükle
2. Ya da **Google Drive** linkini paylaş

Model dosyası: `tft_{trade_type}_{timestamp}.pt`